# 01 - Introduccion a Scikit-Learn

---

## Contenido

1. Que es Scikit-Learn
2. Representacion de datos
   - Datos como tabla
   - Matriz de caracteristicas (X)
   - Vector objetivo (y)
3. API del Estimator
4. Pipeline clasico de Scikit-Learn
5. Ejemplo completo con el dataset Iris

---

## 1. Que es Scikit-Learn

Existen numerosas librerias de Python con implementaciones de algoritmos de Machine Learning. Una de las mas conocidas es [Scikit-Learn](http://scikit-learn.org).

Este paquete provee diferentes algoritmos de ML con **implementaciones eficientes**. Scikit-Learn se caracteriza por dos cosas:

1. Programacion limpia y uniforme para implementar/usar los algoritmos.
2. Buena documentacion en linea.

Por *uniforme* se refiere a que **todos los modelos utilizan el mismo proceso de uso y sintaxis**. De esta forma, cambiar entre algoritmos es muy sencillo.

En este notebook se proporciona una descripcion general de la API de `Scikit-Learn`. Es importante recalcar **los elementos que conforman y la sintaxis de la API**, ya que esto permitira comprender de manera mas profunda el uso de los algoritmos y enfoques de ML en los siguientes modulos.

In [ ]:
# Importaciones necesarias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
sns.set()

print(f'NumPy version: {np.__version__}')
print(f'Pandas version: {pd.__version__}')

import sklearn
print(f'Scikit-Learn version: {sklearn.__version__}')

---

## 2. Representacion de Datos

ML crea modelos a partir de datos: por esa razon, comenzaremos recalcando que es necesario representar los datos para que la computadora los entienda, es decir, en terminos de **tablas de datos** (`numpy.array` o `pandas.DataFrame`). Esta es la misma forma que `Scikit-Learn` emplea para representar y alimentar los datos a sus diferentes funciones (i.e. algoritmos).

### Datos como Tabla

Una tabla es una cuadricula de datos bidimensional:

- **Filas** representan **individuos/muestras** del conjunto de datos ($\mathbf{x}_i$).
- **Columnas** representan variables independientes (caracteristicas) relacionadas con cada uno de estos elementos.

### El dataset Iris

Para ejemplificar, consideramos el famoso conjunto de [flores Iris de Anderson](https://en.wikipedia.org/wiki/Iris_flower_data_set) estudiado por Ronald Fisher en 1936. Este dataset contiene mediciones de 150 flores de tres especies distintas: *setosa*, *versicolor* y *virginica*.

Las 4 caracteristicas medidas son:
- `sepal_length`: longitud del sepalo (cm)
- `sepal_width`: ancho del sepalo (cm)
- `petal_length`: longitud del petalo (cm)
- `petal_width`: ancho del petalo (cm)

In [ ]:
# Cargar el dataset Iris desde Seaborn
iris = sns.load_dataset('iris')

print(f'Dimensiones del dataset: {iris.shape}')
print(f'Numero de muestras: {iris.shape[0]}')
print(f'Numero de columnas: {iris.shape[1]}')
print()
iris.head(10)

In [ ]:
# Informacion general del dataset
iris.info()

In [ ]:
# Estadisticas descriptivas
iris.describe()

In [ ]:
# Distribucion de las especies
print(iris['species'].value_counts())

### Visualizacion exploratoria

Usando un `pairplot` podemos visualizar las relaciones entre todas las variables, coloreando por especie:

In [ ]:
sns.pairplot(iris, hue='species',
             diag_kind='kde',
             palette='bright',
             height=2.5);
plt.suptitle('Pairplot del dataset Iris', y=1.02, fontsize=14);

---

### Matriz de Caracteristicas (X)

La **Matriz de Caracteristicas** es una matriz 2D numerica ($\mathbf{x}_i \in X$).

Por convencion se le suele almacenar en la variable `X`. Su forma (`X.shape`) es `[n_samples, n_features]`. Esta matriz puede ser un arreglo de NumPy o un `DataFrame` de Pandas.

### Vector de la variable dependiente (y)

Ademas de la matriz de caracteristicas `X`, tipicamente tenemos un arreglo de *etiquetas* (para clasificacion) o *valores objetivo* (para regresion), que por convencionalismo denotamos como `y`. Este arreglo es de **1D**, tiene una longitud de `n_samples`, y se representa como un arreglo de NumPy o una `Series` de Pandas.

In [ ]:
# Extraer la matriz de caracteristicas X
X_iris = iris.iloc[:, 0:4]
print(f'Tipo de X: {type(X_iris)}')
print(f'Forma de X: {X_iris.shape}')
print()
X_iris.head()

In [ ]:
# Extraer el vector objetivo y
y_iris = iris['species']
print(f'Tipo de y: {type(y_iris)}')
print(f'Forma de y: {y_iris.shape}')
print()
y_iris.head()

Visualmente, la estructura queda de la siguiente forma:

```
X = [[sepal_length, sepal_width, petal_length, petal_width],   <-- muestra 1
     [sepal_length, sepal_width, petal_length, petal_width],   <-- muestra 2
     ...                                                        
     [sepal_length, sepal_width, petal_length, petal_width]]   <-- muestra n

y = [species_1, species_2, ..., species_n]
```

Donde `X.shape = (150, 4)` y `y.shape = (150,)`.

---

## 3. API del Estimator de Scikit-Learn

Una API (Application Program Interface) especifica como los componentes de un software interactuan. El objeto principal de la API de Scikit-Learn se denomina **Estimator**.

Este objeto **ajusta (fits) un modelo** a un **conjunto de entrenamiento** para **inferir/predecir** alguna propiedad de **nuevos datos**. El modelo puede ser de **clasificacion o regresion**.

Todos los algoritmos de ML en Scikit-Learn son "estimators" con funciones en comun:

| Metodo/Atributo | Descripcion |
|---|---|
| `estimator.fit(X, y)` | Ajusta/entrena el modelo con los datos |
| `estimator.predict(X_new)` | Predice valores para nuevos datos |
| `estimator.score(X, y)` | Evalua el rendimiento del modelo |
| `estimator.coef_` | Coeficientes aprendidos (despues de `fit`) |
| `estimator.intercept_` | Intercepto aprendido (despues de `fit`) |

---

## 4. Pipeline Clasico usando Scikit-Learn

Un "pipeline" en Ciencia de Datos es **una secuencia de pasos de procesamiento y modelado aplicados a datos para un proposito especifico**.

El pipeline clasico para usar el `Estimator` de Scikit-Learn es:

1. **Elegir** la clase apropiada de `estimator` e importarla de Scikit-Learn.
2. **Elegir los parametros** del modelo e instanciarlo.
3. **Organizar los datos** en una matriz de caracteristicas ($X$) y un vector objetivo ($y$).
4. **Ajustar el modelo** a los datos usando `fit()`.
5. **Predecir** nuevas observaciones con `predict()`.

Veamos este pipeline completo con un ejemplo de regresion lineal simple sobre el dataset Iris.

### Ejemplo: Predecir `petal_length` a partir de `sepal_length`

Aunque Iris es un dataset de clasificacion, podemos usarlo para regresion al predecir una variable numerica a partir de otra.

In [ ]:
# PASO 1: Importar la clase del modelo
from sklearn.linear_model import LinearRegression

In [ ]:
# PASO 2: Instanciar el modelo con sus parametros
model = LinearRegression(fit_intercept=True)
print(model)

In [ ]:
# PASO 3: Organizar los datos en X (2D) e y (1D)
X = iris[['sepal_length']].values  # Matriz 2D: (150, 1)
y = iris['petal_length'].values     # Vector 1D: (150,)

print(f'Forma de X: {X.shape}')
print(f'Forma de y: {y.shape}')

In [ ]:
# PASO 4: Ajustar el modelo
model.fit(X, y)

# Parametros aprendidos
print(f'Coeficiente (pendiente): {model.coef_[0]:.4f}')
print(f'Intercepto (ordenada al origen): {model.intercept_:.4f}')
print()
print(f'Ecuacion del modelo: petal_length = {model.intercept_:.4f} + {model.coef_[0]:.4f} * sepal_length')

In [ ]:
# PASO 5: Predecir
x_new = np.linspace(4, 8, 100).reshape(-1, 1)
y_pred = model.predict(x_new)

# Visualizar
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(X, y, color='steelblue', alpha=0.6, label='Datos reales')
ax.plot(x_new, y_pred, color='red', linewidth=2, label='Prediccion del modelo')
ax.set_xlabel('Sepal Length (cm)', fontsize=12)
ax.set_ylabel('Petal Length (cm)', fontsize=12)
ax.set_title('Regresion Lineal Simple: sepal_length vs petal_length', fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Evaluacion rapida con R^2
r2 = model.score(X, y)
print(f'R^2 del modelo: {r2:.4f}')
print(f'Interpretacion: el modelo explica el {r2*100:.1f}% de la varianza de petal_length.')

---

## Resumen

En este notebook aprendimos:

1. **Scikit-Learn** es una libreria de ML con una API uniforme y bien documentada.
2. Los datos se representan como **matriz de caracteristicas** `X` (2D) y **vector objetivo** `y` (1D).
3. La API del **Estimator** provee metodos estandar: `fit()`, `predict()`, `score()`.
4. El **pipeline clasico** sigue 5 pasos: importar, instanciar, organizar datos, ajustar y predecir.
5. Aplicamos estos conceptos con un ejemplo de regresion sobre el dataset Iris.

En el siguiente notebook profundizaremos en la **Regresion Lineal Simple y Multiple**.